In [108]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import os

import networkx as nx
import folium
import osmnx as ox
from matplotlib.pyplot import gray

In [109]:
def geocode_location(prompt_text):
    """
    :return: geo obj, text location from user input
    """
    while True:
        location = input(prompt_text)
        try:
            # TODO: Ensure location is within city limits
            geo = ox.geocode(location)
            return geo, location
        except Exception:
            print("Location not found. Please query again.")

# Source: ChatGPT
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # meters
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi/2)**2 + \
        math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2

    return 2 * R * math.asin(math.sqrt(a))

def loc_prompt():
    """
    :return: start and end geo locations
    """
    geo_start, loc_start = geocode_location("Please enter your starting location: ")
    geo_end, loc_end = geocode_location("Please enter your destination: ")

    print(f"Start Point: {geo_start} \'{loc_start}\'")
    print(f"End Point: {geo_end} \'{loc_end}\'")

    points = [geo_start, geo_end]
    return points

def center(points):
    """
    :param points:
    :return: center point and the correct radius for a plot
    """
    start_lat = points[0][0]
    start_lon = points[0][1]
    end_lat = points[1][0]
    end_lon = points[1][1]

    geo_center = (
        ( float(start_lat)+float(end_lat) ) / 2,
        ( float(start_lon)+float(end_lon) ) / 2
                 )

    # Haversine distance between locations * 1.25 for some bezels for mapping
    bezel_factor = 1.25
    geo_dist = (haversine(start_lat, start_lon, end_lat, end_lon) / 2) * bezel_factor
    # geo_dist = max(geo_dist, 1000)
    geo_dist = round(geo_dist, 2)

    return geo_center, geo_dist, start_lat, start_lon, end_lat, end_lon

In [110]:
G = ox.graph_from_place("Washington, District of Columbia, USA", network_type="walk") # ~2 min runtime

Crime density weight


In [214]:
crime_gdf = gpd.read_file("../../../Data/Crime/Crime_Incidents_in_2025.geojson")

In [215]:
graph = ox.project_graph(G)  # projects to UTM automatically
edges = ox.graph_to_gdfs(graph, nodes=False, edges=True)

crime_gdf = crime_gdf.to_crs(edges.crs)

In [216]:
buffer_dist = 41.5  # HYPERPARAM: buffer distance in meters (41.5-65)
                  # TODO: Incorperate into function to allow user to dial in

edges['geometry_buffer'] = edges.geometry.buffer(buffer_dist)

edges_buffered = edges.set_geometry('geometry_buffer')

# spatial join: crimes within each edge buffer
joined = gpd.sjoin(crime_gdf, edges_buffered, how='left', predicate='within')

In [217]:
crime_counts = joined.groupby(['u', 'v', 'key']).size()

edges['crime_count'] = edges.index.map(crime_counts).fillna(0)
edges['crime_density'] = edges['crime_count'] / edges['length']

In [218]:
alpha = 1  # HYPERPARAM: tune this (0 -> 10+)
            # TODO: Incorperate into function to allow user to dial in


In [219]:
cd_max = edges['crime_density'].max()

edges['crime_density_norm'] = (edges['crime_density'] / cd_max)

edges['crime_weight'] = edges['length'] * (1 + (alpha * edges['crime_density_norm']))

In [220]:
for u, v, k, data in graph.edges(keys=True, data=True):
    if (u, v, k) in edges.index:
        data['crime_weight'] = edges.loc[(u, v, k), 'crime_weight']
    else:
        print('WARING: fallback to length occurred')
        data['crime_weight'] = data['length']  # fallback

User


In [221]:
# Functions have built in print statements
geo_center, geo_dist, start_lat, start_lon, end_lat, end_lon = center(loc_prompt())

# TODO: ask use what time it is, block the time by 'SHIFT' and only account for the crimes in that time chunk

Start Point: (38.8976387, -77.0365528) 'the white house'
End Point: (38.9202726, -76.9982035) '519 Rhode Island Ave NE, Washington, DC 20002'


In [222]:
graph = ox.graph_from_point(geo_center, dist=geo_dist, network_type='walk') # runtime dependant of graph size
# TODO: Do i need this?

In [223]:
origin_node = ox.distance.nearest_nodes(graph, start_lon, start_lat)
destination_node = ox.distance.nearest_nodes(graph, end_lon, end_lat)

route_crime = nx.shortest_path(graph, origin_node, destination_node, weight = 'crime_weight', method = 'dijkstra')
route_short = nx.shortest_path(graph, origin_node, destination_node, weight = 'length', method = 'dijkstra')


Folium graph


In [224]:
from folium.plugins import *

m = folium.Map(
    location = geo_center,
    zoom_start = 12 #TODO: zoom should start dependant on how large the route is
)

In [225]:
edges_gdf = ox.graph_to_gdfs(graph, nodes=False, edges=True)

In [226]:
route_coords_short = [
    (graph.nodes[node]['y'], graph.nodes[node]['x'])
    for node in route_short
]

route_coords_crime = [
    (graph.nodes[node]['y'], graph.nodes[node]['x'])
    for node in route_crime
]

In [227]:
origin_xy = (graph.nodes[origin_node]['x'], graph.nodes[origin_node]['y'])
destination_xy = (graph.nodes[destination_node]['x'], graph.nodes[destination_node]['y'])

In [228]:
ScrollZoomToggler().add_to(m)
folium.PolyLine(route_coords_short, color = 'red', weight = 7, opacity = 0.5).add_to(m)
folium.PolyLine(route_coords_crime, color = 'blue', weight = 3, opacity = 0.5).add_to(m)

folium.Marker(
    location = origin_xy,
    icon = folium.Icon(color='green', icon = 'flag', prefix='fa')
).add_to(m)
folium.Marker(
    location = destination_xy,
    icon = folium.Icon(color='red', icon = 'flag', prefix = 'fa')
).add_to(m)

m

In [229]:
(edges['crime_weight'] - edges['length']).abs().max()

2.2322627718972683

In [238]:
# ChatGPT generated output

def compute_route_metrics(graph, origin_node, destination_node):
    # --- Compute routes ---
    route_crime = nx.shortest_path(
        graph, origin_node, destination_node,
        weight='crime_weight', method='dijkstra'
    )

    route_short = nx.shortest_path(
        graph, origin_node, destination_node,
        weight='length', method='dijkstra'
    )

    # --- Helper function to compute path length ---
    def path_weight(graph, route, weight_attr):
        total = 0
        for u, v in zip(route[:-1], route[1:]):
            edge_data = graph.get_edge_data(u, v)

            # Handle MultiDiGraph (multiple edges)
            if isinstance(edge_data, dict):
                edge_data = min(edge_data.values(), key=lambda x: x.get(weight_attr, float('inf')))

            total += edge_data.get(weight_attr, 0)
        return total

    # --- Compute metrics ---
    crime_route_length_m = path_weight(graph, route_crime, 'length')
    crime_route_weighted = path_weight(graph, route_crime, 'crime_weight')

    short_route_length_m = path_weight(graph, route_short, 'length')
    short_route_weighted = path_weight(graph, route_short, 'crime_weight')

    # --- Print results cleanly ---
    print("\n" + "="*50)
    print("ROUTE COMPARISON")
    print("="*50)

    print("\n[Shortest Distance Route]")
    print(f"  Total Distance (meters):        {short_route_length_m:,.2f}")
    print(f"  Crime-Weighted Distance:        {short_route_weighted:,.2f}")

    print("\n[Safest (Crime-Weighted) Route]")
    print(f"  Total Distance (meters):        {crime_route_length_m:,.2f}")
    print(f"  Crime-Weighted Distance:        {crime_route_weighted:,.2f}")

    print("\n" + "-"*50)
    print("DIFFERENCE")
    print("-"*50)
    print(f"  Extra Distance for Safety (m):  {crime_route_length_m - short_route_length_m:,.2f}")
    print(f"  Crime Reduction:                {short_route_weighted - crime_route_weighted:,.2f}")
    print("="*50 + "\n")

    return {
        "route_short": route_short,
        "route_crime": route_crime,
        "metrics": {
            "shortest": {
                "length_m": short_route_length_m,
                "crime_weight": short_route_weighted
            },
            "safest": {
                "length_m": crime_route_length_m,
                "crime_weight": crime_route_weighted
            }
        }
    }

In [239]:
result = compute_route_metrics(graph, origin_node, destination_node)


ROUTE COMPARISON

[Shortest Distance Route]
  Total Distance (meters):        4,585.23
  Crime-Weighted Distance:        0.00

[Safest (Crime-Weighted) Route]
  Total Distance (meters):        5,223.25
  Crime-Weighted Distance:        0.00

--------------------------------------------------
DIFFERENCE
--------------------------------------------------
  Extra Distance for Safety (m):  638.02
  Crime Reduction:                0.00

